# Actor Holdout Sequence Experiments

Ce notebook reprend le script `actor_holdout_sequence_experiments.py`.

But : rendre l'experience lisible et executable dans Jupyter, avec des explications simples en francais.

Utilisation : executez les cellules dans l'ordre. La derniere cellule lance le script avec des arguments controles.


## Pertinence pour le live streaming

Decision : garde. Teste la generalisation du modele sequence live a des acteurs non vus.

Regle appliquee : l'experience doit aider a entrainer, choisir, calibrer, tester ou executer une prediction en flux video avec seulement les informations disponibles a l'instant courant.


## Pourquoi ce notebook est garde

- Description du script : Actor-held-out sequence model validation.
- Artefacts controles : Actor-held-out sequence validation exists. (`runs/exp_017_actor_holdout_sequence/metrics/actor_holdout_summary.csv`).
- Run par defaut : `runs/exp_017_actor_holdout_sequence`.

Decision : garde, car il correspond a un artefact experimental, une commande de reproduction, ou un audit du catalogue.


## Avant de commencer

- Verifiez que les donnees et les dossiers `runs/` attendus existent.
- Le notebook n'a pas ete execute pendant sa creation.
- Les cellules de lancement creent un nom ou un dossier unique quand cela evite d'ecraser un resultat existant.


In [ ]:
# Compatibilite Jupyter
# Certains scripts utilisent __file__. Dans un notebook, on le definit explicitement.
from pathlib import Path

PROJECT_ROOT = Path.cwd()
__file__ = str(PROJECT_ROOT / "actor_holdout_sequence_experiments.py")


## Importations et configuration

Cette cellule charge les bibliotheques et definit les constantes utilisees par le script.

In [ ]:
import argparse
from pathlib import Path
from types import SimpleNamespace

import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split

from ml_pipeline import ROOT, write_json
from sequence_experiments import append_report, evaluate_catalogue_model, make_run_dir, train_one_model


## Fonction `normalize_for_split`

Cette cellule definit `normalize_for_split`. Elle prepare une partie du script.

In [ ]:
def normalize_for_split(X_raw, meta):
    train_mask = meta["split"].to_numpy() == "train"
    flat = X_raw[train_mask].reshape(-1, X_raw.shape[-1])
    mean = flat.mean(axis=0)
    std = flat.std(axis=0)
    std = np.where(std > 1e-6, std, 1.0)
    return ((X_raw - mean) / std).astype(np.float32), mean, std


## Fonction `make_actor_split`

Cette cellule definit `make_actor_split`. Elle prepare une partie du script.

In [ ]:
def make_actor_split(meta, held_actor, seed):
    videos = meta.groupby("video_id", as_index=False).agg(actor=("actor", "first"), is_danger_clip=("is_danger_clip", "max"))
    test_ids = videos[videos["actor"] == held_actor]["video_id"].to_numpy()
    rest = videos[videos["actor"] != held_actor].copy()
    if len(rest) < 4:
        raise ValueError(f"Not enough non-held actor videos for held actor {held_actor}")
    stratify = rest["is_danger_clip"].astype(int).to_numpy()
    if len(np.unique(stratify)) < 2 or min(np.bincount(stratify)) < 2:
        stratify_arg = None
    else:
        stratify_arg = stratify
    train_ids, val_ids = train_test_split(
        rest["video_id"].to_numpy(),
        test_size=0.25,
        random_state=seed,
        stratify=stratify_arg,
    )
    split = {video_id: "train" for video_id in train_ids}
    split.update({video_id: "val" for video_id in val_ids})
    split.update({video_id: "test" for video_id in test_ids})
    return split


## Fonction `selection_score`

Cette cellule definit `selection_score`. Elle prepare une partie du script.

In [ ]:
def selection_score(row):
    return (
        float(row.get("average_precision", 0) or 0)
        + 0.5 * float(row.get("best_hit_rate", 0) or 0)
        + 0.2 * float(row.get("best_window_precision", 0) or 0)
        - 0.03 * min(float(row.get("best_false_alarms_per_min", 20) or 20), 20.0)
    )


## Fonction `summarize`

Cette cellule definit `summarize`. Elle prepare une partie du script.

In [ ]:
def summarize(metrics):
    h1 = metrics[metrics["horizon_s"].astype(float).eq(1.0)].copy()
    rows = []
    for (held_actor, base_arch, split), group in h1.groupby(["held_actor", "base_architecture", "split"]):
        rows.append(
            {
                "held_actor": held_actor,
                "base_architecture": base_arch,
                "split": split,
                "average_precision": float(group["average_precision"].mean()),
                "roc_auc": float(group["roc_auc"].mean()),
                "best_hit_rate": float(group["best_hit_rate"].mean()),
                "best_false_alarms_per_min": float(group["best_false_alarms_per_min"].mean()),
                "best_window_precision": float(group["best_window_precision"].mean()),
                "selection_score": float(group["selection_score"].dropna().mean()) if group["selection_score"].notna().any() else None,
            }
        )
    return pd.DataFrame(rows)


## Fonction `run`

Cette cellule definit `run`. Elle prepare une partie du script.

In [ ]:
def run(args):
    source_run = Path(args.sequence_run)
    if not source_run.is_absolute():
        source_run = ROOT / source_run
    run_dir = make_run_dir(args.run_name)
    data = np.load(source_run / "features" / "sequence_dataset.npz")
    X_norm = data["X"].astype(np.float32)
    y = data["y"].astype(np.float32)
    mean = data["mean"].astype(np.float32)
    std = data["std"].astype(np.float32)
    X_raw = X_norm * std.reshape(1, 1, -1) + mean.reshape(1, 1, -1)
    base_meta = pd.read_csv(source_run / "features" / "sequence_index.csv")
    videos = pd.read_csv(ROOT / "annotations" / "videos.csv")[["video_id", "actor"]]
    base_meta = base_meta.merge(videos, on="video_id", how="left")
    actors = [actor for actor in sorted(base_meta["actor"].dropna().unique()) if actor]
    specs = [
        ("tcn_aug_bce", "tcn", True, "bce"),
        ("tcn_noaug_bce", "tcn", False, "bce"),
        ("tcn_aug_focal", "tcn", True, "focal"),
        ("tcn_noaug_focal", "tcn", False, "focal"),
        ("cnn1d_aug_focal", "cnn1d", True, "focal"),
        ("gru_noaug_bce", "gru", False, "bce"),
        ("cnn_gru_aug_bce", "cnn_gru", True, "bce"),
        ("lstm_aug_bce", "lstm", True, "bce"),
    ]
    if args.quick:
        specs = specs[:3]
    write_json(
        run_dir / "config.json",
        {
            "sequence_run": str(source_run),
            "actors": actors,
            "architectures": [spec[0] for spec in specs],
            "epochs": args.epochs,
            "patience": args.patience,
            "policy": "held actor is test; remaining actor videos split into train/val",
        },
    )
    device = torch.device("cuda" if torch.cuda.is_available() and args.device == "auto" else args.device)
    all_metrics = []
    all_history = []
    split_rows = []
    for held_actor in actors:
        split = make_actor_split(base_meta, held_actor, args.seed)
        meta = base_meta.copy()
        meta["split"] = meta["video_id"].map(split)
        meta.to_csv(run_dir / "features" / f"actor_holdout_{held_actor}_index.csv", index=False)
        split_counts = meta.groupby("split")["video_id"].nunique().to_dict()
        danger_counts = meta.groupby(["split", "is_danger_clip"])["video_id"].nunique().to_dict()
        split_rows.append({"held_actor": held_actor, **split_counts})
        X, split_mean, split_std = normalize_for_split(X_raw, meta)
        np.savez_compressed(run_dir / "features" / f"normalizer_holdout_{held_actor}.npz", mean=split_mean, std=split_std)
        for base_name, kind, augment, loss in specs:
            model_name = f"holdout_{held_actor}_{base_name}"
            print(f"training {model_name}")
            model_args = SimpleNamespace(
                seed=args.seed,
                batch_size=args.batch_size,
                lr=args.lr,
                weight_decay=args.weight_decay,
                epochs=args.epochs,
                patience=args.patience,
                loss=loss,
                label_smoothing=args.label_smoothing,
                focal_gamma=args.focal_gamma,
            )
            model, history, train_time_s, model_size_bytes = train_one_model(model_name, kind, augment, X, y, meta, run_dir, model_args, device)
            for row in history:
                row["held_actor"] = held_actor
                row["base_architecture"] = base_name
                row["loss"] = loss
            all_history.extend(history)
            rows, _ = evaluate_catalogue_model(
                model_name,
                model,
                X,
                y,
                meta,
                run_dir,
                device,
                train_time_s,
                model_size_bytes,
                args.batch_size,
                loss,
            )
            for row in rows:
                row["held_actor"] = held_actor
                row["base_architecture"] = base_name
                row["selection_score"] = selection_score(row) if row["split"] == "val" and float(row["horizon_s"]) == 1.0 else None
            all_metrics.extend(rows)
            pd.DataFrame(all_metrics).to_csv(run_dir / "metrics" / "actor_holdout_metrics.csv", index=False)
            pd.DataFrame(all_history).to_csv(run_dir / "metrics" / "actor_holdout_training_history.csv", index=False)
    metrics = pd.DataFrame(all_metrics)
    metrics.to_csv(run_dir / "metrics" / "actor_holdout_metrics.csv", index=False)
    pd.DataFrame(all_history).to_csv(run_dir / "metrics" / "actor_holdout_training_history.csv", index=False)
    pd.DataFrame(split_rows).to_csv(run_dir / "metrics" / "actor_holdout_split_counts.csv", index=False)
    summary = summarize(metrics)
    summary.to_csv(run_dir / "metrics" / "actor_holdout_summary.csv", index=False)
    lines = ["# Actor-Held-Out Sequence Validation", ""]
    lines.append("Each actor is used as held-out test once. The remaining actor's videos are split into train/val by parent video.")
    lines.append("")
    lines.append("| held actor | architecture | split | AP | ROC AUC | hit | FA/min | precision |")
    lines.append("|---|---|---|---:|---:|---:|---:|---:|")
    for _, row in summary.sort_values(["held_actor", "split", "average_precision"], ascending=[True, True, False]).iterrows():
        lines.append(
            f"| {row['held_actor']} | {row['base_architecture']} | {row['split']} | {row['average_precision']:.3f} | {row['roc_auc']:.3f} | {row['best_hit_rate']:.3f} | {row['best_false_alarms_per_min']:.3f} | {row['best_window_precision']:.3f} |"
        )
    (run_dir / "actor_holdout_summary.md").write_text("\n".join(lines) + "\n", encoding="utf-8")
    append_report(run_dir, "Actor-Held-Out Completion", f"- Actors: `{actors}`\n- Summary: `{run_dir / 'actor_holdout_summary.md'}`")
    print(run_dir)


## Point d'entree principal

Cette cellule definit `main`. Elle prepare une partie du script.

In [ ]:
def main():
    parser = argparse.ArgumentParser(description="Actor-held-out sequence model validation.")
    parser.add_argument("--sequence-run", default="runs/exp_008_sequence_len60_catalogue")
    parser.add_argument("--run-name", default="exp_017_actor_holdout_sequence")
    parser.add_argument("--epochs", type=int, default=25)
    parser.add_argument("--patience", type=int, default=5)
    parser.add_argument("--batch-size", type=int, default=128)
    parser.add_argument("--lr", type=float, default=1e-3)
    parser.add_argument("--weight-decay", type=float, default=1e-4)
    parser.add_argument("--label-smoothing", type=float, default=0.05)
    parser.add_argument("--focal-gamma", type=float, default=2.0)
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--device", default="auto")
    parser.add_argument("--quick", action="store_true")
    args = parser.parse_args()
    run(args)


## Lancer le script

Cette cellule lance le `main()` avec des arguments adaptes au notebook.

In [ ]:
# Lancement du script
# Modifiez NOTEBOOK_ARGS si vous voulez changer les options.
from datetime import datetime
import sys

RUN_NAME_BASE = "exp_017_actor_holdout_sequence_notebook"
RUN_NAME = f"{RUN_NAME_BASE}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
NOTEBOOK_ARGS = ["--run-name", RUN_NAME]

ancien_argv = sys.argv[:]
sys.argv = ["actor_holdout_sequence_experiments.py"] + NOTEBOOK_ARGS
try:
    print("Arguments utilises :", sys.argv[1:])
    main()
finally:
    sys.argv = ancien_argv
